# `region` 03: related-feature analysis

            **Purpose:** identify predictors that represent the same concept, form a
            hierarchy, share a missingness process or plausibly interact with
            `region`.

            ## Relationships selected in advance

            - `region_code` — The named and coded fields overlap but are not simple duplicates.
- `lga` — Each LGA maps to one region in the supplied data.
- `basin` — Hydrological basins cross administrative regions.
- `longitude` — Coordinates should broadly agree with named region.
- `latitude` — Coordinates should broadly agree with named region.


In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_stage_directory():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (
            (candidate / "data" / "TrainingSetValues.csv").exists()
            and (candidate / "src" / "source_data_validation.py").exists()
        ):
            return candidate
    raise FileNotFoundError("Could not locate the stage-1-pump-it-up directory.")


stage_directory = find_stage_directory()
source_directory = str((stage_directory / "src").resolve())
if source_directory not in sys.path:
    sys.path.insert(0, source_directory)

from predictor_audit import (
    analysis_categories,
    categorical_summary,
    categorical_target_profile,
    category_frequency_table,
    numeric_summary,
    numeric_target_summary,
    related_feature_summary,
    sentinel_mask,
    source_blank_mask,
    text_normalisation_summary,
)
from source_data_validation import (
    validate_aligned_ids,
    validate_label_frame,
    validate_raw_feature_schema,
)

data_directory = stage_directory / "data"
training_features = pd.read_csv(
    data_directory / "TrainingSetValues.csv",
    keep_default_na=False,
)
training_labels = pd.read_csv(
    data_directory / "TrainingSetLabels.csv",
    keep_default_na=False,
)
test_features = pd.read_csv(
    data_directory / "TestSetValues.csv",
    keep_default_na=False,
)

validate_raw_feature_schema(training_features)
validate_raw_feature_schema(test_features)
validate_label_frame(training_labels)
validate_aligned_ids(training_features, training_labels)

training_data = training_features.merge(
    training_labels,
    on="id",
    validate="one_to_one",
)

feature = 'region'
feature_metadata = {'order': 12, 'name': 'region', 'audit_type': 'category', 'role': 'candidate', 'disposition': 'retain as interpretable geographic back-off', 'finding': 'All 21 levels are covered and functional rates differ substantially by region.', 'decision': 'Retain and add an LGA/region-grouped validation sensitivity check.', 'risk': 'Random validation can reward geographic memorisation.', 'related': [{'feature': 'region_code', 'reason': 'The named and coded fields overlap but are not simple duplicates.'}, {'feature': 'lga', 'reason': 'Each LGA maps to one region in the supplied data.'}, {'feature': 'basin', 'reason': 'Hydrological basins cross administrative regions.'}, {'feature': 'longitude', 'reason': 'Coordinates should broadly agree with named region.'}, {'feature': 'latitude', 'reason': 'Coordinates should broadly agree with named region.'}]}
feature_types = {'amount_tsh': 'numeric', 'date_recorded': 'date', 'funder': 'high-cardinality-category', 'gps_height': 'numeric', 'installer': 'high-cardinality-category', 'longitude': 'coordinate', 'latitude': 'coordinate', 'wpt_name': 'high-cardinality-category', 'num_private': 'numeric', 'basin': 'category', 'subvillage': 'high-cardinality-category', 'region': 'category', 'region_code': 'category', 'district_code': 'category', 'lga': 'category', 'ward': 'high-cardinality-category', 'population': 'numeric', 'public_meeting': 'binary', 'recorded_by': 'constant', 'scheme_management': 'category', 'scheme_name': 'high-cardinality-category', 'permit': 'binary', 'construction_year': 'year', 'extraction_type': 'category', 'extraction_type_group': 'category', 'extraction_type_class': 'category', 'management': 'category', 'management_group': 'category', 'payment': 'category', 'payment_type': 'category', 'water_quality': 'category', 'quality_group': 'category', 'quantity': 'category', 'quantity_group': 'category', 'source': 'category', 'source_type': 'category', 'source_class': 'category', 'waterpoint_type': 'category', 'waterpoint_type_group': 'category'}
assert feature in training_features.columns
print(
    f"Validated {len(training_features):,} training rows and "
    f"{len(test_features):,} test rows for {feature}."
)


Validated 59,400 training rows and 14,850 test rows for region.


In [2]:
relationship_inventory = pd.DataFrame(feature_metadata["related"])
display(relationship_inventory)

relationship_evidence = related_feature_summary(
    training_features,
    feature,
    feature_metadata["audit_type"],
    feature_metadata["related"],
    feature_types,
)
display(relationship_evidence)


,feature,reason
0,region_code,The named and coded fields overlap but are not...
1,lga,Each LGA maps to one region in the supplied data.
2,basin,Hydrological basins cross administrative regions.
3,longitude,Coordinates should broadly agree with named re...
4,latitude,Coordinates should broadly agree with named re...


,primary,related,measure,association,complete rows,primary levels,related levels,forward modal purity (%),reverse modal purity (%),relationship rationale
0,region,region_code,bias-corrected Cramer's V,0.9981,59400,21,27,95.64,99.79,The named and coded fields overlap but are not...
1,region,lga,bias-corrected Cramer's V,0.9991,59400,21,125,31.79,100.00,Each LGA maps to one region in the supplied data.
2,region,basin,bias-corrected Cramer's V,0.7671,59400,21,9,73.42,39.45,Hydrological basins cross administrative regions.
3,region,longitude,correlation ratio (eta),0.6271,59400,21,57516,NaN,NaN,Coordinates should broadly agree with named re...
4,region,latitude,correlation ratio (eta),0.9692,59400,21,57517,NaN,NaN,Coordinates should broadly agree with named re...


## Discussion and modelling consequence

The relationships above were nominated before inspecting the pairwise
coefficients. A strong association can mean useful interaction, hierarchy,
shared collection behaviour or redundancy; it is not a reason to keep both
fields automatically.

For `region`, carry the relationships into controlled ablations
and fit every learned grouping or encoding inside the training fold. The
current provisional disposition remains: **retain as interpretable geographic back-off**.
